<a href="https://colab.research.google.com/github/asakicode/git-study-AIM-/blob/main/BDAI%EC%A0%84%EC%B2%98%EB%A6%AC_3%EC%A3%BC%EC%B0%A8_%EA%B3%BC%EC%A0%9C.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import seaborn as sns
import numpy as np

# 1. 데이터 불러오기
df = sns.load_dataset('titanic')

# 필요한 컬럼만 추출 (나이, 성별, 등급, 요금, 생존여부)
titanic = df[['survived', 'pclass', 'sex', 'age', 'fare', 'embarked']].copy()

print("--- [타이타닉 원본 상위 5행] ---")
print(titanic.head())
print("\n")

# ---------------------------------------------------------
# 2. pd.cut vs pd.qcut (연령대/요금 구간화)
# ---------------------------------------------------------
# pd.cut: 나이를 기준으로 생애주기 나누기 (0-15: 미성년, 15-60: 성인, 60-100: 노년)
titanic['age_group'] = pd.cut(titanic['age'], bins=[0, 15, 60, 100], labels=['Child', 'Adult', 'Elderly'])

# pd.qcut: 요금(fare)을 기준으로 4등분 (가장 싼 25%, ..., 가장 비싼 25%)
titanic['fare_level'] = pd.qcut(titanic['fare'], q=4, labels=['Bronze', 'Silver', 'Gold', 'VIP'])

print("--- [구간화 결과 확인] ---")
print(titanic[['age', 'age_group', 'fare', 'fare_level']].head(10))
print("\n")

# ---------------------------------------------------------
# 3. pivot & crosstab (데이터 재구조화)
# ---------------------------------------------------------
# pivot_table: 성별과 선실 등급에 따른 평균 생존율
# (pivot은 중복값 처리가 안 되므로 집계 기능이 있는 pivot_table을 주로 씁니다)
survival_pivot = titanic.pivot_table(index='sex', columns='pclass', values='survived', aggfunc='mean')

# crosstab: 성별에 따른 승선 항구(embarked) 빈도수
embark_cross = pd.crosstab(titanic['sex'], titanic['embarked'])

print("--- [성별/등급별 평균 생존율] ---")
print(survival_pivot)
print("\n")

# -----------------------------------------------------
# 4. 결측치와 MCAR (Missing Value)
# -------------------------------------------------------
print(f"나이 데이터 결측치 개수: {titanic['age'].isnull().sum()}")

# MCAR 여부 생각하기:
# 결측치 채우기 (나이의 중앙값으로 채우기)
titanic['age_filled'] = titanic['age'].fillna(titanic['age'].median())

print("\n--- [결측치 처리 후 확인] ---")
print(f"채우기 후 결측치 개수: {titanic['age_filled'].isnull().sum()}")

--- [타이타닉 원본 상위 5행] ---
   survived  pclass     sex   age     fare embarked
0         0       3    male  22.0   7.2500        S
1         1       1  female  38.0  71.2833        C
2         1       3  female  26.0   7.9250        S
3         1       1  female  35.0  53.1000        S
4         0       3    male  35.0   8.0500        S


--- [구간화 결과 확인] ---
    age age_group     fare fare_level
0  22.0     Adult   7.2500     Bronze
1  38.0     Adult  71.2833        VIP
2  26.0     Adult   7.9250     Silver
3  35.0     Adult  53.1000        VIP
4  35.0     Adult   8.0500     Silver
5   NaN       NaN   8.4583     Silver
6  54.0     Adult  51.8625        VIP
7   2.0     Child  21.0750       Gold
8  27.0     Adult  11.1333     Silver
9  14.0     Child  30.0708       Gold


--- [성별/등급별 평균 생존율] ---
pclass         1         2         3
sex                                 
female  0.968085  0.921053  0.500000
male    0.368852  0.157407  0.135447


나이 데이터 결측치 개수: 177

--- [결측치 처리 후 확인] ---
채우기 후 